# 02 — Feature Engineering

**Purpose:** construct leakage-safe lifecycle snapshots from the canonical daily dataset. Future outcomes are targets only; current scoring snapshots contain no `target_*` columns.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))


In [2]:
from game_performance.config import SETTINGS
from game_performance.features import build_all_historical_snapshots, categorical_features, model_features
from game_performance.io import load_dataframe, save_artifact

df = load_dataframe(SETTINGS.processed_dir / "clean_game_daily.pkl")


In [3]:
historical_snapshots = build_all_historical_snapshots(df, SETTINGS.horizon_strategy)
summary = {key: len(frame) for key, frame in historical_snapshots.items()}
summary


{(14, 14): 9301}

In [4]:
example_key = next(iter(historical_snapshots))
example = historical_snapshots[example_key]
print("Example configuration:", example_key)
print("Categorical features:", categorical_features(example))
print("Model feature count:", len(model_features(example)))
example.head()


Example configuration: (14, 14)
Categorical features: ['SystemName', 'TournamentName', 'Launch_Month', 'Launch_Weekday', 'Observation_End_Month', 'Observation_End_Weekday', 'Clients_Size_Band', 'Stakes_Size_Band']
Model feature count: 114


,Game_ID,SystemName,TournamentName,First_Observed_Date,Launch_Month,Launch_Weekday,Observation_End_Month,Observation_End_Weekday,Observation_End_DayOfMonth,Observation_End_WeekOfYear,...,target_stakes,target_clients_week_1,target_clients_week_2,target_stakes_week_1,target_stakes_week_2,Prediction_Week,actual_clients_week_3,actual_clients_week_4,actual_stakes_week_3,actual_stakes_week_4
0,AmusnetSlots::AmusnetS-100 Bulky Fruits,AmusnetSlots,AmusnetS-100 Bulky Fruits,2026-02-01,February,Sunday,February,Saturday,14,7,...,190018.0,364,479,145948.0,190018.0,4,364,479,145948.0,190018.0
1,AmusnetSlots::AmusnetS-100 Cats,AmusnetSlots,AmusnetS-100 Cats,2026-02-01,February,Sunday,February,Saturday,14,7,...,1114534.0,6593,4914,1863829.0,1114534.0,4,6593,4914,1863829.0,1114534.0
2,AmusnetSlots::AmusnetS-100 Power Hot,AmusnetSlots,AmusnetS-100 Power Hot,2026-02-01,February,Sunday,February,Saturday,14,7,...,6685104.0,8494,8437,2504075.0,6685104.0,4,8494,8437,2504075.0,6685104.0
3,AmusnetSlots::AmusnetS-2 Dragons,AmusnetSlots,AmusnetS-2 Dragons,2026-02-01,February,Sunday,February,Saturday,14,7,...,115100.6,669,859,102502.0,115100.6,4,669,859,102502.0,115100.6
4,AmusnetSlots::AmusnetS-20 Bulky Fruits,AmusnetSlots,AmusnetS-20 Bulky Fruits,2026-02-01,February,Sunday,February,Saturday,14,7,...,845772.0,2428,2299,436716.6,845772.0,4,2428,2299,436716.6,845772.0


In [5]:
save_artifact(historical_snapshots, SETTINGS.feature_dir / "historical_snapshots.joblib")
print("Saved feature snapshots")


Saved feature snapshots
